<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/Exercises_XP_W6Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- **Printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted:**
```
index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | quick        |  3909
    3 | brown        |  2829
    4 | fox          |  4618
    5 | jumps        |  7369
    6 | over         |  2058
    7 | the          |  1996
    8 | lazy         | 11130
    9 | dog          |  3899
   10 | .            |  1012
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0
```
- **Document the padding choice you made and why it fits the sentence length:**
  I chose `max_length=24` for padding. This length is sufficient to accommodate the tokenized `sample_sentence` ("The quick brown fox jumps over the lazy dog.") along with the special `[CLS]` and `[SEP]` tokens, and then pad the remainder with `[PAD]` tokens up to 24 positions. The original sentence tokenizes to 12 tokens ([CLS] + 10 word tokens + [SEP]), so 12 positions are padded to reach 24.

In [12]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The quick brown fox jumps over the lazy dog."
print(sample_sentence)

The quick brown fox jumps over the lazy dog.


In [14]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | quick        |  4248
    3 | brown        |  2829
    4 | fox          |  4419
    5 | jumps        | 14523
    6 | over         |  2058
    7 | the          |  1996
    8 | lazy         | 13971
    9 | dog          |  3899
   10 | .            |  1012
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]

### Exercise 1 reflection
- **How [CLS] and [SEP] behave inside the encoder:**
  The `[CLS]` (Classification) token is placed at the very beginning of the sequence. It doesn't have an initial semantic meaning, but through the self-attention mechanism, it interacts equally with all other tokens at each encoder layer. At the output, its final vector representation aggregates the entire context of the sentence, making it the ideal vector for global classification tasks.

  The `[SEP]` (Separator) token serves as an explicit boundary between two distinct sentences (e.g., in question-answering tasks). In the encoder, it helps attention mechanisms know where the first segment ends and the second begins, while also acting as a focal point to indicate the end of a textual proposition.

- **How the attention mask hides padded positions from self-attention:**
  The attention mask functions as a binary filter applied directly during the calculation of attention scores before the Softmax function step. For each position corresponding to a padding token (`[PAD]`), the mask assigns a value of 0 (or an extremely large negative value like -∞ or -10000 in matrix calculations). When the Softmax function is applied to these modified scores, the exponential of these very large negative numbers tends towards 0 (e^-∞ → 0). Consequently, the attention weight given to padding positions becomes strictly zero (0%), which mathematically forces the model to completely ignore these tokens and concentrate its computation solely on the actual words of the sequence.

Comportement de [CLS] (Classification) : Placé au tout début de la séquence, le jeton [CLS] n'a pas de sens sémantique initial. Cependant, grâce au mécanisme de self-attention, il interagit de manière égale avec tous les autres jetons à chaque couche de l'encodeur. À la sortie, sa représentation vectorielle finale agrège l'ensemble du contexte de la phrase, ce qui en fait le vecteur idéal pour les tâches de classification globale.

Comportement de [SEP] (Separator) : Ce jeton sert de frontière explicite entre deux phrases distinctes (par exemple dans des tâches de question-réponse). Dans l'encodeur, il permet aux mécanismes d'attention de savoir où se termine le premier segment et où commence le second, tout en agissant comme un point de focalisation pour indiquer la fin d'une proposition textuelle.

 TODO: Explain how the attention mask hides padded positions from self-attention.Le masque d'attention fonctionne comme un filtre binaire appliqué directement lors du calcul des scores d'attention avant l'étape de la fonction Softmax.Pour chaque position correspondant à un jeton de remplissage ([PAD]), le masque attribue une valeur de 0 (ou une valeur négative extrêmement grande comme $-\infty$ ou $-10000$ dans le calcul des matrices).Lorsque la fonction Softmax est appliquée sur ces scores modifiés, l'exponentielle de ces très grands nombres négatifs tend vers 0 ($e^{-\infty} \to 0$). Par conséquent, le poids d'attention accordé aux positions de padding devient strictement nul ($0\%$), ce qui force mathématiquement le modèle à ignorer complètement ces jetons et à concentrer son calcul uniquement sur les vrais mots de la séquence.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [15]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This movie is absolutely fantastic and I loved every minute of it!"
prediction = sentiment_pipeline(sentence)
prediction

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998838901519775}]

### Exercise 2 reflection
- **Does the predicted label match your expectation? Why or why not?**
  Yes, the predicted label 'POSITIVE' perfectly matches my expectation. The sentence "This movie is absolutely fantastic and I loved every minute of it!" contains strongly positive sentiment words like "fantastic" and "loved," making a positive classification highly appropriate.

- **How confident is the model and what does the score tell you?**
  The model is extremely confident, with a score of 0.9998838901519775. This score, being very close to 1, indicates that the model is almost certain about its positive classification. It suggests that the input sentence strongly aligns with the patterns of positive sentiment it learned during its training on the SST-2 dataset.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].to(self.device),
            'attention_mask': encoding['attention_mask'].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        # Ensure model is in evaluation mode
        self.model.eval()

        # Preprocess the text to get input tensors
        inputs = self.preprocess(text)

        with torch.no_grad():
            # Perform a forward pass
            outputs = self.model(**inputs)

        # Get logits and apply softmax to get probabilities
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).squeeze().tolist()

        # Get the predicted label index
        predicted_index = torch.argmax(logits, dim=1).item()

        # Map the index to the label (0 for NEGATIVE, 1 for POSITIVE for SST-2)
        label_map = {0: 'NEGATIVE', 1: 'POSITIVE'}
        predicted_label = label_map[predicted_index]
        confidence_score = probabilities[predicted_index]

        return {'label': predicted_label, 'score': confidence_score}

In [23]:
analyzer = BERTSentimentAnalyzer()
samples = [
     "This is a positive statement",
     "This is an non-positive statement",
     "I hate this movie, it's terrible.",
     "What a wonderful day!"
]
for text in samples:
     print(f"Text: {text}")
     print(f"Prediction: {analyzer.predict(text)}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This is a positive statement
Prediction: {'label': 'POSITIVE', 'score': 0.9998756647109985}

Text: This is an non-positive statement
Prediction: {'label': 'NEGATIVE', 'score': 0.9356645345687866}

Text: I hate this movie, it's terrible.
Prediction: {'label': 'NEGATIVE', 'score': 0.9996496438980103}

Text: What a wonderful day!
Prediction: {'label': 'POSITIVE', 'score': 0.9998810291290283}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [25]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def recognize(self, text: str):
        self.model.eval() # Set model to evaluation mode

        # Tokenize input text
        tokens = self.tokenizer.tokenize(self.tokenizer.cls_token + " " + text + " " + self.tokenizer.sep_token)
        # Get input IDs and attention mask
        input_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)

        # Convert to PyTorch tensors and move to device
        input_ids = torch.tensor([input_ids]).to(self.device)
        attention_mask = torch.tensor([attention_mask]).to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask=attention_mask)

        # Get predicted token labels (argmax over logits)
        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()
        predicted_labels = [self.model.config.id2label[p_id] for p_id in predictions]

        # Align tokens with original text and merge subwords
        word_tokens = []
        word_labels = []
        current_word = ""
        current_label = "O"
        start_index = 0

        # The tokenizer adds [CLS] and [SEP] tokens at the beginning and end, respectively.
        # We need to skip these for proper alignment with the original text.
        # Start from the second token (after [CLS]) and go up to the second-to-last (before [SEP]).
        for i in range(1, len(tokens) - 1):
            token = tokens[i]
            label = predicted_labels[i]

            if token.startswith("##"):
                current_word += token[2:]
            else:
                if current_word:
                    word_tokens.append(current_word)
                    word_labels.append(current_label)
                current_word = token
                current_label = label

        if current_word:
            word_tokens.append(current_word)
            word_labels.append(current_label)

        # Extract entities
        entities = []
        current_entity = {"text": "", "entity": "", "start": -1, "end": -1}

        for i, (word, label) in enumerate(zip(word_tokens, word_labels)):
            if label.startswith("B-"):
                if current_entity["text"]:
                    entities.append(current_entity)
                current_entity = {"text": word, "entity": label[2:], "start": -1, "end": -1}
            elif label.startswith("I-") and current_entity["entity"] == label[2:]:
                current_entity["text"] += " " + word
            else:
                if current_entity["text"]:
                    entities.append(current_entity)
                current_entity = {"text": "", "entity": "", "start": -1, "end": -1}

        if current_entity["text"]:
            entities.append(current_entity)

        # Calculate character start and end indices for each entity in the original text
        current_char_index = 0
        final_entities = []
        for entity_data in entities:
            entity_text_parts = entity_data["text"].split(" ")

            # Find the first occurrence of the entity's first word in the remaining text
            temp_text = text[current_char_index:]
            first_word = entity_text_parts[0]

            # Adjust for case sensitivity if necessary, but BERT is typically cased or uncased already
            # and we're dealing with original text for index finding.
            start_pos = temp_text.find(first_word)

            if start_pos != -1:
                start_absolute_index = current_char_index + start_pos
                end_absolute_index = start_absolute_index + len(entity_data["text"]) - (len(entity_text_parts) - 1) # Adjust for spaces

                # Verify the actual substring matches to avoid false positives (e.g., if 'New' is in 'New York' but also appears elsewhere)
                if text[start_absolute_index : end_absolute_index].replace(' ','') == entity_data['text'].replace(' ',''):
                    entity_data["start"] = start_absolute_index
                    entity_data["end"] = end_absolute_index
                    final_entities.append(entity_data)
                    current_char_index = end_absolute_index # Move the search window past this found entity


        return final_entities

In [26]:
# Instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin in Menlo Park, California. It is now headquartered in Mountain View."
recognized_entities = ner.recognize(sample_text)
print(recognized_entities)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'Google', 'entity': 'ORG', 'start': 0, 'end': 6}, {'text': 'California', 'entity': 'LOC', 'start': 64, 'end': 74}]


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only (bidirectional attention)| TODO |
| Primary purpose | Understanding context (NLU)| Generating text (NLG) |
| Typical use cases | Sentiment analysis, NER, Q&A, text classification |Text generation, summarization, translation, dialogue systems|
| Strengths | Excellent for contextual understanding, fine-tuning | Highly fluent and coherent text generation, few-shot learning |
| Weaknesses | Cannot directly generate text (needs a decoder) | Less effective at deep contextual understanding due to unidirectional nature |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. **How BERT encodes queries and documents:**
   BERT encodes both the user's query and the corpus of documents (or chunks of documents) into dense numerical vectors called embeddings. For a query, BERT processes the input text to produce a contextualized embedding for each token. Typically, the embedding corresponding to the `[CLS]` token (or an average/pooling of all token embeddings) is used as the fixed-size vector representation of the entire query. Similarly, for documents, each document (or smaller, fixed-size chunks of documents) is passed through BERT to generate a representative embedding. These embeddings capture the semantic meaning and context of the text, allowing for a rich comparison between texts.
2. **Explain how those embeddings are stored and searched in a vector database:**
   Once BERT generates these high-dimensional vector embeddings for both the query and each document (or document chunk), these document embeddings are stored in a specialized database known as a vector database (e.g., Pinecone, Milvus, Weaviate). When a user query arrives, it's first transformed into its own embedding using the same BERT model. This query embedding is then used to search the vector database. The database employs efficient nearest-neighbor search algorithms (like Annoy, Faiss, HNSW) to quickly find document embeddings that are most similar to the query embedding. Similarity is typically measured using distance metrics such as cosine similarity or Euclidean distance, identifying documents whose semantic content is closest to the query.
3. **Outline how the retrieved passages are handed to a generative model like GPT:**
   After the retrieval step, the top-k most relevant document passages (or chunks) found by the vector database are passed to a generative language model, such as GPT. This is typically done by concatenating the original user query with the retrieved passages, often separated by special tokens or specific formatting (e.g., `"Question: [query]\nContext: [passage1] [passage2] ...\nAnswer:"`). The generative model then uses this augmented input as its prompt. By providing relevant context directly within the prompt, the GPT model is guided to generate a more accurate, informed, and contextually grounded answer, rather than relying solely on its internal parametric knowledge.
4. **Provide a concrete application example (industry or product) where RAG with BERT makes sense:**
   A concrete application where RAG with BERT excels is in **customer support chatbots for large enterprises**. Imagine a company with extensive documentation, FAQs, product manuals, and internal knowledge bases. When a customer asks a complex question (e.g., "How do I troubleshoot a specific error code on my router model XYZ?"), a traditional chatbot might fail or provide generic answers. With RAG, BERT would first encode the customer's query. This query embedding would then be used to retrieve the most relevant sections or documents from the vast knowledge base stored in a vector database. These retrieved passages, containing the exact troubleshooting steps for router model XYZ and the error code, are then fed along with the original query to a generative model like GPT. GPT can then synthesize a precise, accurate, and human-like answer based on the provided, up-to-date context, significantly improving customer satisfaction and reducing the workload on human agents.